In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
df = spark.read.table('fintech360.business_transformation.upi_data')
df.limit(5).display()

In [0]:

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# ============================================================
# 1. CONFIGURATION
# ============================================================
source_table = "fintech360.business_transformation.upi_data"
target_table = "fintech360.gold.upi_data"

spark.sql("CREATE SCHEMA IF NOT EXISTS fintech360.gold")

# ============================================================
# 2. READ BUSINESS-TRANSFORMED DATA
# ============================================================
df = spark.read.table(source_table)

# ============================================================
# 3. DEDUPLICATE THE INCOMING BATCH
# Keep the latest business-transformed record per transaction
# ============================================================
window_spec = Window.partitionBy("upi_txn_id").orderBy(
    F.col("business_transformed_at").desc_nulls_last(),
    F.col("updated_at").cast("long").desc_nulls_last()
)

df = (
    df.withColumn("_gold_row_number", F.row_number().over(window_spec))
      .filter(F.col("_gold_row_number") == 1)
      .drop("_gold_row_number")
)

# ============================================================
# 4. STANDARDIZE CORE FIELDS
# ============================================================
df = (
    df
    .withColumn("gold_transaction_id", F.trim(F.col("upi_txn_id")))
    .withColumn("gold_customer_id", F.col("customer_id").cast("long"))
    .withColumn("gold_transaction_amount", F.col("business_transaction_amount").cast("decimal(18,2)"))
    .withColumn("gold_currency", F.upper(F.trim(F.col("business_currency"))))
    .withColumn("gold_transaction_status", F.upper(F.trim(F.col("business_transaction_status"))))
    .withColumn("gold_merchant_category", F.upper(F.trim(F.col("business_merchant_category"))))
    .withColumn("gold_upi_app", F.upper(F.trim(F.col("business_upi_app"))))
    .withColumn("gold_device_os", F.upper(F.trim(F.col("business_device_os"))))
    .withColumn("gold_account_type", F.upper(F.trim(F.col("business_account_type"))))
    .withColumn("gold_payer_vpa", F.lower(F.trim(F.col("business_payer_vpa"))))
    .withColumn("gold_payee_vpa", F.lower(F.trim(F.col("business_payee_vpa"))))
)

# ============================================================
# 5. TRANSACTION DATE AND TIME DIMENSIONS
# ============================================================
df = (
    df
    .withColumn(
        "gold_transaction_timestamp",
        F.coalesce(
            F.to_timestamp(F.col("business_transaction_datetime")),
            F.to_timestamp(F.from_unixtime(F.col("transaction_ts").cast("long")))
        )
    )
    .withColumn("gold_transaction_date", F.to_date("gold_transaction_timestamp"))
    .withColumn("gold_transaction_month", F.date_format("gold_transaction_timestamp", "yyyy-MM"))
    .withColumn("gold_transaction_year", F.year("gold_transaction_timestamp"))
    .withColumn("gold_transaction_hour", F.hour("gold_transaction_timestamp"))
    .withColumn("gold_day_of_week", F.date_format("gold_transaction_timestamp", "EEEE"))
    .withColumn(
        "gold_is_weekend",
        F.when(F.dayofweek("gold_transaction_timestamp").isin(1, 7), 1).otherwise(0)
    )
)

# ============================================================
# 6. TRANSACTION OUTCOME AND REFUND MEASURES
# ============================================================
df = (
    df
    .withColumn(
        "gold_is_successful",
        F.when(F.col("gold_transaction_status") == "SUCCESS", 1).otherwise(0)
    )
    .withColumn(
        "gold_is_failed",
        F.when(
            F.col("gold_transaction_status").isin("FAILED", "FAILURE", "DECLINED", "TIMEOUT"),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_is_refund",
        F.when(F.col("business_is_refund").cast("int") == 1, 1).otherwise(0)
    )
    .withColumn(
        "gold_successful_transaction_amount",
        F.when(
            F.col("gold_is_successful") == 1,
            F.coalesce(F.col("gold_transaction_amount"), F.lit(0))
        ).otherwise(F.lit(0)).cast("decimal(18,2)")
    )
    .withColumn(
        "gold_failed_transaction_amount",
        F.when(
            F.col("gold_is_failed") == 1,
            F.coalesce(F.col("gold_transaction_amount"), F.lit(0))
        ).otherwise(F.lit(0)).cast("decimal(18,2)")
    )
    .withColumn(
        "gold_successful_refund_amount",
        F.when(
            (F.col("gold_is_successful") == 1) & (F.col("gold_is_refund") == 1),
            F.coalesce(F.col("gold_transaction_amount"), F.lit(0))
        ).otherwise(F.lit(0)).cast("decimal(18,2)")
    )
    .withColumn(
        "gold_successful_non_refund_amount",
        F.when(
            (F.col("gold_is_successful") == 1) & (F.col("gold_is_refund") == 0),
            F.coalesce(F.col("gold_transaction_amount"), F.lit(0))
        ).otherwise(F.lit(0)).cast("decimal(18,2)")
    )
)

# ============================================================
# 7. SPENDING AND TRANSACTION SEGMENTS
# ============================================================
df = (
    df
    .withColumn(
        "gold_transaction_amount_band",
        F.when(F.col("gold_transaction_amount").isNull(), "UNKNOWN")
         .when(F.col("gold_transaction_amount") < 100, "BELOW_100")
         .when(F.col("gold_transaction_amount") < 500, "100_TO_499")
         .when(F.col("gold_transaction_amount") < 1000, "500_TO_999")
         .when(F.col("gold_transaction_amount") < 5000, "1000_TO_4999")
         .otherwise("5000_PLUS")
    )
    .withColumn(
        "gold_spending_group",
        F.when(F.col("gold_is_refund") == 1, "REFUND")
         .when(F.col("gold_merchant_category").isin("FOOD", "GROCERY"), "ESSENTIAL_SPENDING")
         .when(F.col("gold_merchant_category").isin("HEALTHCARE", "EDUCATION"), "NECESSITY_SPENDING")
         .when(F.col("gold_merchant_category").isin("ENTERTAINMENT", "SHOPPING", "TRAVEL"), "DISCRETIONARY_SPENDING")
         .when(F.col("gold_merchant_category").isNull(), "UNKNOWN_CATEGORY")
         .otherwise("OTHER_SPENDING")
    )
    .withColumn(
        "gold_value_segment",
        F.when(F.col("gold_transaction_amount").isNull(), "UNKNOWN")
         .when(F.col("gold_transaction_amount") >= 5000, "VERY_HIGH_VALUE")
         .when(F.col("gold_transaction_amount") >= 1000, "HIGH_VALUE")
         .when(F.col("gold_transaction_amount") >= 500, "MEDIUM_VALUE")
         .otherwise("LOW_VALUE")
    )
)

# ============================================================
# 8. CUSTOMER AND TRANSACTION REVIEW SIGNALS
# These are screening indicators, not fraud determinations.
# ============================================================
df = (
    df
    .withColumn(
        "gold_high_value_flag",
        F.when(F.col("gold_transaction_amount") >= 1000, 1).otherwise(0)
    )
    .withColumn(
        "gold_same_vpa_flag",
        F.when(
            F.col("gold_payer_vpa").isNotNull() &
            (F.col("gold_payer_vpa") == F.col("gold_payee_vpa")),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_missing_rrn_flag",
        F.when(
            (F.col("gold_is_successful") == 1) &
            (F.col("rrn").isNull() | (F.trim(F.col("rrn")) == "")),
            1
        ).otherwise(0)
    )
    .withColumn(
        "gold_review_signal",
        F.when(F.col("gold_transaction_id").isNull(), "MISSING_TRANSACTION_ID")
         .when(F.col("gold_customer_id").isNull(), "MISSING_CUSTOMER_ID")
         .when(
             F.col("gold_transaction_amount").isNull() |
             (F.col("gold_transaction_amount") < 0),
             "INVALID_TRANSACTION_AMOUNT"
         )
         .when(F.col("gold_same_vpa_flag") == 1, "SAME_PAYER_PAYEE_VPA")
         .when(F.col("gold_missing_rrn_flag") == 1, "SUCCESS_WITHOUT_RRN")
         .when(F.col("gold_is_failed") == 1, "FAILED_TRANSACTION")
         .when(F.col("gold_is_refund") == 1, "REFUND_TRANSACTION")
         .when(F.col("gold_high_value_flag") == 1, "HIGH_VALUE_TRANSACTION")
         .otherwise("NO_REVIEW_SIGNAL")
    )
)

# ============================================================
# 9. DATA QUALITY FLAGS AND GOLD METADATA
# ============================================================
df = (
    df
    .withColumn(
        "gold_data_quality_status",
        F.when(F.col("gold_transaction_id").isNull(), "INVALID")
         .when(F.col("gold_customer_id").isNull(), "INVALID")
         .when(
             F.col("gold_transaction_amount").isNull() |
             (F.col("gold_transaction_amount") < 0),
             "INVALID"
         )
         .otherwise("VALID")
    )
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_table", F.lit(source_table))
)

# ============================================================
# 10. SELECT GOLD OUTPUT
# Keep original columns and append Gold-specific features.
# ============================================================
gold_df = df

# ============================================================
# 11. APPEND TO GOLD TABLE
# Exclude transaction IDs already present in Gold to make
# normal reruns less likely to append duplicate transactions.
# ============================================================
if spark.catalog.tableExists(target_table):
    existing_ids = (
        spark.read.table(target_table)
        .select(F.col("upi_txn_id").alias("_existing_upi_txn_id"))
        .filter(F.col("_existing_upi_txn_id").isNotNull())
        .dropDuplicates()
    )

    gold_df = (
        gold_df.join(
            existing_ids,
            gold_df["upi_txn_id"] == existing_ids["_existing_upi_txn_id"],
            "left_anti"
        )
    )

    (
        gold_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )
else:
    (
        gold_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(target_table)
    )

print(f"Gold data successfully written to {target_table}")

# ============================================================
# 12. VALIDATE GOLD TABLE
# ============================================================
spark.sql(f"""
SELECT
    COUNT(*) AS total_transactions,
    COUNT(DISTINCT upi_txn_id) AS unique_transactions,
    SUM(gold_is_successful) AS successful_transactions,
    SUM(gold_is_failed) AS failed_transactions,
    SUM(gold_is_refund) AS refund_transactions,
    SUM(gold_successful_transaction_amount) AS successful_amount,
    SUM(gold_failed_transaction_amount) AS failed_amount,
    SUM(gold_successful_refund_amount) AS successful_refund_amount,
    SUM(gold_high_value_flag) AS high_value_transactions
FROM {target_table}
""").show(truncate=False)

# Preview
spark.read.table(target_table).orderBy(
    F.col("gold_transaction_timestamp").desc_nulls_last()
).show(10, truncate=False)
